# 02. Compressor fit

목표: development identity의 원본 임베딩만 사용해 PCA와 Faiss PQ를 학습하고, 모델과 학습 요약을 run artifact로 고정합니다. 성공 기준은 test/calibration 누수 없이 모델 파일의 hash와 학습 표본 수가 기록되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 01이 완료되고 원본 임베딩 수가 확인된 run에서만 실행합니다. 중단되면 02 전체를 다시 실행해 새 attempt를 만들며, attempt 번호가 붙은 기존 모델 artifact는 덮어쓰지 않습니다. manifest/development split이 바뀌면 00부터 새 run을 시작합니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import RunStore, resolve_active_run

EXECUTE_STAGE = False
RUN_ROOT = PROJECT_ROOT / 'runs'
RUN_DIR = resolve_active_run(RUN_ROOT)


## Plan

- Join DB embeddings to manifest paths and retain only `split == development`.
- Fit PCA in its retrieval space and PQ as an auxiliary non-pgvector code baseline.
- Save attempt-specific models and record model/profile metadata.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file()),
}
preflight


## Execute and record

PCA/PQ 학습 표본은 반드시 development split으로 제한합니다. PQ code는 `LargeBinary`용 보조 artifact이며 pgvector 검색 벡터로 취급하지 않습니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import numpy as np
    import pandas as pd
    from research.compression import PCACompressor, PQCompressor
    from research.database import create_database_engine, load_database_settings, session_scope
    from research.database.models import Embedding512, Image

    run, run_manifest = attach_run(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts('01_arcface_embedding_extraction')
    config = run_manifest['config']
    manifest = pd.read_csv(PROJECT_ROOT / config['dataset']['manifest_path'])
    development_paths = {
        str((PROJECT_ROOT / Path(str(path))).resolve()) if not Path(str(path)).is_absolute() else str(Path(str(path)).resolve())
        for path in manifest.loc[manifest['split'].eq('development'), 'image_path']
    }
    engine = create_database_engine(load_database_settings())
    with session_scope(engine) as session:
        db_rows = (session.query(Embedding512, Image).join(Image, Embedding512.image_id == Image.id)
                   .filter(Embedding512.vector_type == 'arcface', Embedding512.run_uid == run.run_id).all())
        vectors = [np.asarray(embedding.embedding, dtype=np.float32) for embedding, image in db_rows
                   if str(Path(image.image_path).resolve()) in development_paths]
    if not vectors:
        raise ValueError('No development ArcFace embeddings were found for this run.')
    matrix = np.stack(vectors)
    pca_cfg = config['compression'].get('pca', {})
    pq_cfg = config['compression'].get('pq', {})
    pca = PCACompressor(int(pca_cfg.get('n_components', 256)), random_state=int(config['protocol'].get('split_seed', 42))).fit(matrix)
    pq = PQCompressor(matrix.shape[1], m=int(pq_cfg.get('m', 16)), nbits=int(pq_cfg.get('nbits', 8))).fit(matrix)
    with run.phase('02_compressor_fit') as phase:
        suffix = f'A{phase.attempt:03d}'
        pca_source = pca.save(phase.attempt_dir / f'pca_256_{suffix}.joblib')
        pq_source = pq.save(phase.attempt_dir / f'pq_{suffix}.faiss')
        pca_artifact = phase.publish_artifact(pca_source)
        pq_artifact = phase.publish_artifact(pq_source)
        summary = {
            'fit_split': 'development', 'fit_count': int(len(matrix)), 'source_dim': int(matrix.shape[1]),
            'pca': {'artifact': str(pca_artifact.relative_to(run.run_dir)), 'n_components': pca.n_components},
            'pq': {'artifact': str(pq_artifact.relative_to(run.run_dir)), 'm': pq.m, 'nbits': pq.nbits, 'pgvector_searchable': False},
        }
        summary_source = phase.attempt_dir / f'compressor_summary_{suffix}.json'
        summary_source.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        phase.publish_artifact(summary_source)
        phase.record_counts(development_vectors=len(matrix))
    result = {'status': 'completed', 'run_id': run.run_id, **summary}
result


## Next step

fit count, PCA 차원, PQ 파라미터를 확인합니다. PQ 학습 실패 시 표본 수와 `m/nbits`를 기록한 뒤 설정 변경은 00의 새 run으로 수행합니다.
